In [1]:
from datasets import load_from_disk

tiny_story_instruct_dataset = load_from_disk('./tiny-stories-instruct-dataset')

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tiny_story_instruct_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 21755681
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 218380
    })
})

In [3]:
END_OF_TEXT = '<|endoftext|>'

def get_story(dataset, index):
    story = []
    start_ind = index
    while True:
        if index >= len(dataset):
            break
        story_line = dataset[index]['text']
        if story_line.strip() == END_OF_TEXT:
            break
        if story_line.strip():
            if index == start_ind and story_line.startswith('Story:'):
                # Remove 'Story:' prefix (if present) in the first line of the Story
                story.append(story_line.removeprefix('Story:'))
            else:
                story.append(story_line)
        index += 1

    return ' '.join(story)

def transform_dataset(dataset):
    # Runtime of approx 4-5 mins in my Mac (CPU) with 32 Gigs of RAM.
    log_intervals = 1000000
    count = 0
    total = len(dataset)
    story_summary_dataset = []
    current_story = ''
    current_summary = ''
    faulty_rows = 0

    for ind, line in enumerate(dataset):
        if line['text'].startswith('Summary:'):
            current_summary = line['text'].removeprefix('Summary: ')
        if line['text'].startswith('Story:'):
            current_story = get_story(dataset, ind)
        if line['text'].strip() == END_OF_TEXT:
            if current_story == '' or current_summary == '':
                print(f'Either Story or Summary is empty: {faulty_rows}')
                faulty_rows+=1
            else:
                story_summary_dataset.append((current_story, current_summary))
            # reset the story and summary since the current item is completed
            current_story = ''
            current_summary = ''

        count += 1
        if count % log_intervals == 0:
            print(f'Count: {count} / {total}, Percentage: {(count/total) * 100.0}')
    
    return story_summary_dataset

tiny_story_summary_pairs = transform_dataset(tiny_story_instruct_dataset['train'])

Count: 1000000 / 21755681, Percentage: 4.596500564611147
Count: 2000000 / 21755681, Percentage: 9.193001129222294
Either Story or Summary is empty: 0
Count: 3000000 / 21755681, Percentage: 13.789501693833442
Count: 4000000 / 21755681, Percentage: 18.386002258444588
Count: 5000000 / 21755681, Percentage: 22.982502823055732
Either Story or Summary is empty: 1
Count: 6000000 / 21755681, Percentage: 27.579003387666884
Count: 7000000 / 21755681, Percentage: 32.17550395227803
Count: 8000000 / 21755681, Percentage: 36.772004516889176
Count: 9000000 / 21755681, Percentage: 41.368505081500324
Count: 10000000 / 21755681, Percentage: 45.965005646111464
Count: 11000000 / 21755681, Percentage: 50.56150621072262
Count: 12000000 / 21755681, Percentage: 55.15800677533377
Count: 13000000 / 21755681, Percentage: 59.75450733994491
Count: 14000000 / 21755681, Percentage: 64.35100790455606
Count: 15000000 / 21755681, Percentage: 68.9475084691672
Count: 16000000 / 21755681, Percentage: 73.54400903377835
Cou

In [ ]:
len(tiny_story_summary_pairs)

In [4]:
tiny_story_summary_pairs[1010]

(' Once upon a time, there was a big ship on the sea. The ship was so big that it could carry lots of things. One day, the ship saw a small boat in the water. The boat was so soft that it looked like a pillow. The ship offered to help the boat and carry it on its back. The boat was so happy and said thank you to the ship. The ship was happy to help the boat and they became good friends.',
 'A big ship helps a small boat and they become friends.')

In [5]:
from transformers import AutoTokenizer, TextStreamer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
tokenizer.pad_token = tokenizer.eos_token

In [7]:
# Running method in parallel and hence processing individual elements. 
def format_and_tokenize_single_dataset(story_summary_pair):
    story = story_summary_pair[0].strip()
    summary = story_summary_pair[1].strip()

    formatted_dataset = f"Story: {story} \nSummary: {summary}{tokenizer.eos_token}"
    formatted_tokenized_dataset = tokenizer.encode(formatted_dataset)

    return formatted_tokenized_dataset

In [8]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

tasks = (delayed(format_and_tokenize_single_dataset)(pair) for pair in tiny_story_summary_pairs)

# Run in parallel
tokenized_story_summary_pairs = list(tqdm(
    Parallel(n_jobs=-1, return_as="generator", batch_size=10000)(tasks), 
    total=len(tiny_story_summary_pairs), 
    desc="Tokenizing"
))

Tokenizing: 100%|██████████| 2476530/2476530 [02:59<00:00, 13784.36it/s]


In [9]:
print(len(tokenized_story_summary_pairs))

2476530


In [10]:
print(tokenized_story_summary_pairs[1010])

[11605, 25, 4874, 2402, 257, 640, 11, 612, 373, 257, 1263, 4074, 319, 262, 5417, 13, 383, 4074, 373, 523, 1263, 326, 340, 714, 3283, 6041, 286, 1243, 13, 1881, 1110, 11, 262, 4074, 2497, 257, 1402, 8848, 287, 262, 1660, 13, 383, 8848, 373, 523, 2705, 326, 340, 3114, 588, 257, 28774, 13, 383, 4074, 4438, 284, 1037, 262, 8848, 290, 3283, 340, 319, 663, 736, 13, 383, 8848, 373, 523, 3772, 290, 531, 5875, 345, 284, 262, 4074, 13, 383, 4074, 373, 3772, 284, 1037, 262, 8848, 290, 484, 2627, 922, 2460, 13, 220, 198, 22093, 25, 317, 1263, 4074, 5419, 257, 1402, 8848, 290, 484, 1716, 2460, 13, 50256]


In [11]:
from datasets import Dataset

def prepare_tokenized_dataset(tokenized_list):
    items = 100000
    tokenized_trimmed_list = tokenized_list[:items]
    tokenized_dataset_dict = {'input_ids': tokenized_trimmed_list}

    hf_dataset = Dataset.from_dict(tokenized_dataset_dict)

    return hf_dataset

In [13]:
hf_tokenized_dataset = prepare_tokenized_dataset(tokenized_story_summary_pairs)

In [14]:
hf_tokenized_dataset

Dataset({
    features: ['input_ids'],
    num_rows: 100000
})

In [15]:
hf_tokenized_dataset.save_to_disk('./tiny-story-summary-tokenized/')

Saving the dataset (1/1 shards): 100%|██████████| 100000/100000 [00:00<00:00, 197548.86 examples/s]
